In [1]:
import pandas as pd
import numpy as np


import xgboost as xgb

from sklearn.metrics import classification_report, roc_auc_score

import matplotlib.pyplot as plt

In [2]:
print(xgb.__version__)

3.2.0


In [5]:
df = pd.read_csv("../data/processed/features_2023.csv")

```
unlike logistic regression, XGBoost doesn't care about feature scale, multicollinearity, or even raw categorical text in some cases, so we use the .csv file rather than the scaled version
```

In [8]:
print(f"Shape of the dataset: {df.shape}")

Shape of the dataset: (11089, 33)


In [10]:
# Split races 

test_races = ["Singapore", "Monza"]

train_df = df[~df["RaceName"].isin(test_races)]
test_df = df[df["RaceName"].isin(test_races)]

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (9079, 33)
Test shape: (2010, 33)


In [12]:
# Seperate Features(X) from target (y)

exclude_cols = [
    "Pitted", "Driver", "RaceName", "Compound",
    "IsAccurate", "FastF1Generated", "IsPersonalBest"
]

feature_col = [col for col in train_df.columns if col not in exclude_cols]

X_train = train_df[feature_col]
y_train = train_df["Pitted"]

X_test =test_df[feature_col]
y_test = test_df["Pitted"]


print("Feature columns", feature_col, "\n\n")
print(f"X_train shape: {X_train.shape}\n\n")
print(f"y_train shape: {y_train.shape}")

Feature columns ['Time', 'LapTime', 'LapNumber', 'Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife', 'FreshTyre', 'LapStartTime', 'Position', 'CompoundEncoded', 'LapTimeDelta', 'LapTimeRolling', 'DegradationFromStintStart', 'TotalLaps', 'RacePctComplete', 'LapsRemaining', 'IsLateRace'] 


X_train shape: (9079, 26)


y_train shape: (9079,)


 ---

```
Time, LapStartTime, Sector1/2/3SessionTime, LapNumber
→ all measuring essentially "race progress" in different units
→ including all of them doesn't help, just adds redundant noise 
  and makes the model slightly slower without real benefit

```

In [16]:
redundant_time_cols = [
    "Time", "LapStartTime",
    "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime",
    "LapNumber",  # superseded by RacePctComplete
    "LapTime"   # superseded by LapTimeRolling3

]

feature_col = [col for col in feature_col if col not in redundant_time_cols]

X_train =train_df[feature_col]
X_test = test_df[feature_col]

print("Cleaned feature columns:", feature_col)
print(f"\nX_train shape: {X_train.shape}")

Cleaned feature columns: ['Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife', 'FreshTyre', 'Position', 'CompoundEncoded', 'LapTimeDelta', 'LapTimeRolling', 'DegradationFromStintStart', 'TotalLaps', 'RacePctComplete', 'LapsRemaining', 'IsLateRace']

X_train shape: (9079, 19)


In [18]:
# Training the model 
# Handle class imbalance (33 : 1)
# scale_pos_weight tells XGBoost how muh more to weight the rare class

scale_pos_weight = (y_train == 0).sum() /(y_train ==1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

scale_pos_weight: 32.1


In [20]:
model_xgb = xgb.XGBClassifier(
    n_estimators = 300, # no of decision trees to build.
    max_depth = 6, # how deep each individual tree can grow
    learning_rate = 0.05,  # how much each new tree corrects the mistakes of previous trees 
    subsample = 0.8,
    colsample_bytree = 0.8, # each tree only see 80% of the rows+ cols (randomly choosen) this adds randomness helps prevent overfitting
    scale_pos_weight = scale_pos_weight,
    eval_metric = "auc",
    random_state = 42,
    n_jobs = -1

)
model_xgb.fit(X_train, y_train)
print("Model trained")

Model trained


In [22]:
importance = pd.DataFrame({
    "feature": feature_col,
    "Importance": model_xgb.feature_importances_
}).sort_values("Importance", ascending = False)

print(importance)

                      feature  Importance
0                       Stint    0.116429
3                 Sector3Time    0.072800
17              LapsRemaining    0.071726
2                 Sector2Time    0.067442
8                    TyreLife    0.063173
11            CompoundEncoded    0.061207
16            RacePctComplete    0.052505
14  DegradationFromStintStart    0.051745
4                     SpeedI1    0.051357
10                   Position    0.050622
12               LapTimeDelta    0.045856
13             LapTimeRolling    0.045024
6                     SpeedFL    0.042347
1                 Sector1Time    0.042255
15                  TotalLaps    0.040735
5                     SpeedI2    0.040071
7                     SpeedST    0.031447
18                 IsLateRace    0.029901
9                   FreshTyre    0.023355


In [24]:
for col in ["SpeedFL", "Sector3Time"]:
    print(f"\n{col} by Pitted status:")
    print(df.groupby("Pitted")[col].describe()[["mean", "std", "min", "max"]])


SpeedFL by Pitted status:
              mean        std   min    max
Pitted                                    
0       262.724823  30.732757  59.0  322.0
1       259.252308  32.789043  85.0  311.0

Sector3Time by Pitted status:
             mean       std     min     max
Pitted                                     
0       27.496765  5.042594  20.059  59.997
1       27.123265  4.173735  20.444  44.319


In [26]:
# Predictions

# get predictions on the unseen test races
y_pred_xgb = model_xgb.predict(X_test)
y_pred_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

print("XGBoost Performance (Singapore + Monza, unseen):\n")
print(classification_report(y_test, y_pred_xgb))

auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
print(f"ROC-AUC: {auc_xgb:.4f}")

XGBoost Performance (Singapore + Monza, unseen):

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1959
           1       0.08      0.08      0.08        51

    accuracy                           0.95      2010
   macro avg       0.53      0.53      0.53      2010
weighted avg       0.95      0.95      0.95      2010

ROC-AUC: 0.7874


```
Why this might be happening:
11. XGBoost might be overfitting to the training races. With max_depth=6 and n_estimators=300, it has a lot of capacity to memorize patterns specific to the 8 training races that don't generalize to Singapore/Monza's different characteristics (street circuit vs. high-speed circuit).


2. The severe class imbalance (33:1) combined with XGBoost's flexibility might cause it to learn overly specific, narrow rules from the training data that don't transfer well — while Logistic Regression's simplicity (one global linear boundary) might actually generalize more robustly here, even though it's "less powerful" in theory.


3. Hyperparameters haven't been tuned at all yet — we used reasonable defaults, but never did any tuning (grid search, cross-validation) to find what actually works best for THIS specific problem

```

 ----

In [25]:
print(type(model_xgb))
print(y_pred_xgb[:20])  # first 20 predictions
print(y_test[:20].values)  # first 20 actual values

<class 'xgboost.sklearn.XGBClassifier'>
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]


In [27]:
# Check 1: Did Pitted accidentally end up IN our features?
print("Pitted" in feature_col)

# Check 2: Are there any features that perfectly correlate with Pitted?

correlations_with_target = train_df[feature_col + ["Pitted"]].corr()["Pitted"].sort_values(ascending=False)
print(correlations_with_target)

False
Pitted                       1.000000
LapTimeDelta                 0.242671
Sector3Time                  0.203882
DegradationFromStintStart    0.152609
LapTimeRolling3              0.080928
TyreLife                     0.078751
Sector2Time                  0.070588
TrackStatus                  0.038763
LapsRemaining                0.026426
Sector1Time                  0.018893
Position                     0.011366
FreshTyre                    0.004001
TotalLaps                   -0.008015
SpeedFL                     -0.012594
RacePctComplete             -0.032384
SpeedI1                     -0.036142
SpeedI2                     -0.037075
SpeedST                     -0.049433
IsLateRace                  -0.061702
CompoundEncoded             -0.067462
Stint                       -0.086404
Name: Pitted, dtype: float64


In [29]:
# Are there any duplicate rows between train and test?
overlap = pd.merge(train_df, test_df, how='inner')
print("Number of overlapping rows:", len(overlap))

# Double check race separation is clean
print("Train races:", train_df["RaceName"].unique())
print("Test races:", test_df["RaceName"].unique())
print("Any race appears in both?", 
      bool(set(train_df["RaceName"].unique()) & set(test_df["RaceName"].unique())))

Number of overlapping rows: 0
Train races: ['Abu Dhabi' 'Australia' 'Bahrain' 'Hungary' 'Monaco' 'Saudi Arabia'
 'Silverstone' 'Spain']
Test races: ['Monza' 'Singapore']
Any race appears in both? False


---

In [30]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_xgb)
print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives:  {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives:  {cm[1][1]}")

Confusion Matrix:
[[1911   48]
 [  47    4]]

True Negatives:  1911
False Positives: 48
False Negatives: 47
True Positives:  4


In [35]:
# TrackStatus leaks pit stop information — remove it completely
updated_feature_col = [col for col in feature_col if col != "TrackStatus"]

X_train = train_df[updated_feature_col]
X_test = test_df[updated_feature_col]

print("Updated feature columns:", updated_feature_col)
print(f"\nX_train shape: {X_train.shape}")

Updated feature columns: ['Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife', 'FreshTyre', 'Position', 'CompoundEncoded', 'LapTimeDelta', 'LapTimeRolling3', 'DegradationFromStintStart', 'TotalLaps', 'RacePctComplete', 'LapsRemaining', 'IsLateRace']

X_train shape: (9079, 19)


In [38]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

model_xgb = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    random_state=42,
    n_jobs=-1
)

model_xgb.fit(X_train, y_train)

y_pred_xgb = model_xgb.predict(X_test)
y_pred_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

print("XGBoost Performance WITHOUT TrackStatus (Singapore + Monza, unseen):\n")
print(classification_report(y_test, y_pred_xgb))

auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
print(f"ROC-AUC: {auc_xgb:.4f}")

scale_pos_weight: 32.1
XGBoost Performance WITHOUT TrackStatus (Singapore + Monza, unseen):

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1959
           1       0.08      0.08      0.08        51

    accuracy                           0.95      2010
   macro avg       0.53      0.53      0.53      2010
weighted avg       0.95      0.95      0.95      2010

ROC-AUC: 0.7874
